In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import imagehash
import hashlib
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed

In [5]:
df = pd.read_csv('train.csv')
df

,id,label,label_number,label_folder_name,file_name
0,R_1,Recyclable,0,0_Recyclable,R_1.jpg
1,R_2,Recyclable,0,0_Recyclable,R_2.jpg
2,R_3,Recyclable,0,0_Recyclable,R_3.jpg
3,R_4,Recyclable,0,0_Recyclable,R_4.jpg
4,R_5,Recyclable,0,0_Recyclable,R_5.jpg
...,...,...,...,...,...
26519,O_12563,Organic,2,2_Organic,O_12563.jpg
26520,O_12564,Organic,2,2_Organic,O_12564.jpg
26521,O_12565,Organic,2,2_Organic,O_12565.jpg
26522,O_12566,Organic,2,2_Organic,O_12566.jpg


In [6]:
df["file_path"] = "train/" + df["label_folder_name"] + "/" + df["file_name"]
df.head()

,id,label,label_number,label_folder_name,file_name,file_path
0,R_1,Recyclable,0,0_Recyclable,R_1.jpg,train/0_Recyclable/R_1.jpg
1,R_2,Recyclable,0,0_Recyclable,R_2.jpg,train/0_Recyclable/R_2.jpg
2,R_3,Recyclable,0,0_Recyclable,R_3.jpg,train/0_Recyclable/R_3.jpg
3,R_4,Recyclable,0,0_Recyclable,R_4.jpg,train/0_Recyclable/R_4.jpg
4,R_5,Recyclable,0,0_Recyclable,R_5.jpg,train/0_Recyclable/R_5.jpg


In [7]:
sample_per_label = (
    df[df['label'].isin(['Recyclable', 'Organic', 'Electronic'])]
    .groupby('label', as_index=False)
    .first()
)
sample_per_label

,label,id,label_number,label_folder_name,file_name,file_path
0,Electronic,E_1,1,1_Electronic,battery_0.jpg,train/1_Electronic/battery_0.jpg
1,Organic,O_1,2,2_Organic,O_1.jpg,train/2_Organic/O_1.jpg
2,Recyclable,R_1,0,0_Recyclable,R_1.jpg,train/0_Recyclable/R_1.jpg


## Preprocessing & Cleaning

#### Cek gambar rusak

In [8]:
def cek_gambar(filepath):
    """
    Mengecek gambar yang corrupt atau tidak

    :param filepath: path file yang akan di cek
    :return: True or False
    """

    try:
        with Image.open(filepath) as img:
            img.verify()
        with Image.open(filepath) as img:
            img.load()
        return True
    except Exception:
        return False

df['not_corrupt'] = df['file_path'].apply(cek_gambar)

total_rusak = (~df['not_corrupt']).sum()
print(f"Jumlah file corrupt: {total_rusak} dari len(df_gambar) = {len(df)}")


Jumlah file corrupt: 0 dari len(df_gambar) = 26524


In [10]:
help(cek_gambar)

Help on function cek_gambar in module __main__:

cek_gambar(filepath)
    Mengecek gambar yang corrupt atau tidak

    :param filepath: path file yang akan di cek
    :return: True or False



#### Cek Duplikat

In [72]:
def md5_hash(filepath):
    """
    Menghitung MD5 hash dari file biner.

    :param filepath: path file yang akan di cek
    :return: MD5 hash
    """
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception:
        return None


In [73]:
paths = df['file_path'].tolist()
hashes = [None] * len(paths)

# parallel work supaya cepet euy
with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_idx = {executor.submit(md5_hash, p): i for i, p in enumerate(paths)}
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            hashes[idx] = future.result()
        except:
            pass

df['md5'] = hashes
df_clean = df.dropna(subset=['md5']).copy()
print(f"Gambar dengan hash valid: {len(df_clean)} dari {len(df)}")


Gambar dengan hash valid: 26524 dari 26524


In [74]:
md5_hashes = df_clean['md5'].values
filepaths = df_clean['file_path'].values

seen = {}  # md5 -> (index, filepath)
indices_to_keep = []
duplicate_records = []

for idx, (h, fp) in enumerate(zip(md5_hashes, filepaths)):
    if h not in seen:
        seen[h] = (idx, fp)
        indices_to_keep.append(idx)
    else:
        kept_idx, kept_fp = seen[h]
        duplicate_records.append({
            'kept_file': kept_fp,
            'duplicate_file': fp,
            'md5_hash': h
        })

In [75]:
df_duplicates = pd.DataFrame(duplicate_records,
                             columns=['kept_file', 'duplicate_file', 'md5_hash'])

df_dedup = df_clean.iloc[indices_to_keep].copy()
df_dedup.drop(columns=['md5'], inplace=True)

print(f"\nGambar setelah deduplikasi: {len(df_dedup)} dari {len(df_clean)}")
print(f"Jumlah pasangan duplikat (exact): {len(df_duplicates)}")


Gambar setelah deduplikasi: 26462 dari 26524
Jumlah pasangan duplikat (exact): 62


In [76]:
print("Daftar file duplikat (10 teratas):")
df_duplicates


Daftar file duplikat (10 teratas):


,kept_file,duplicate_file,md5_hash
0,train/0_Recyclable/R_38.jpg,train/0_Recyclable/R_1277.jpg,79bd35d90c441cc30c504a386c74602c
1,train/0_Recyclable/R_8486.jpg,train/0_Recyclable/R_9205.jpg,08a364c3cede63a2273820590b06498e
2,train/0_Recyclable/R_8715.jpg,train/0_Recyclable/R_9275.jpg,02b6ec5bc7c76efed16027bea19fd211
3,train/0_Recyclable/R_8442.jpg,train/0_Recyclable/R_9284.jpg,f68c12198de0306eac34db4d1ae67ae9
4,train/0_Recyclable/R_8577.jpg,train/0_Recyclable/R_9410.jpg,793daccac5f6354f2b2865ce9369edd4
...,...,...,...
57,train/2_Organic/O_6683.jpg,train/2_Organic/O_10077.jpg,28f1232d4bd2e18a3dfa29cc3ce6a853
58,train/2_Organic/O_6766.jpg,train/2_Organic/O_10162.jpg,ff8d67f71f4cac204417e65ffcce34c3
59,train/2_Organic/O_6765.jpg,train/2_Organic/O_10266.jpg,ba625da23810a2e5179928dd1f0fd614
60,train/2_Organic/O_6746.jpg,train/2_Organic/O_10316.jpg,62679eaad00c2efda4058ce52e84ff89


#### Cek Blur

In [77]:
def variansi_laplace(filepath):
    """
    Hitung skor blur menggunakan Variance of Laplacian.
    :param filepath:
    :return: varians laplace
    """
    try:
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        # Konvolusi Laplacian dan hitung variansinya
        laplacian = cv2.Laplacian(img, cv2.CV_64F)
        return laplacian.var()
    except Exception:
        return None

In [78]:
paths = df_dedup['file_path'].tolist()
blur_scores = [None] * len(paths)

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_idx = {executor.submit(variansi_laplace(), p): i for i, p in enumerate(paths)}
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            blur_scores[idx] = future.result()
        except:
            pass

df_dedup['blur_score'] = blur_scores
df_blur_clean = df_dedup.dropna(subset=['blur_score']).copy()

In [79]:
# Statistik skor blur
print("\nStatistik skor blur (semakin kecil = semakin buram):")
print(df_blur_clean['blur_score'].describe())


Statistik skor blur (semakin kecil = semakin buram):
count     26462.000000
mean       2844.366214
std        4564.099436
min           0.487546
25%         461.615633
50%        1444.910256
75%        3436.790704
max      105360.382425
Name: blur_score, dtype: float64


In [80]:
# Tentukan threshold: misalnya persentil 1% terendah (sangat buram)
threshold = np.percentile(df_blur_clean['blur_score'], 1)
print(f"\nThreshold persentil ke-1: {threshold:.2f}")


Threshold persentil ke-1: 4.34


In [81]:
df_blur_clean['is_blurry'] = df_blur_clean['blur_score'] < threshold
num_blurry = df_blur_clean['is_blurry'].sum()
print(f"Jumlah gambar terindikasi sangat buram: {num_blurry} dari {len(df_blur_clean)}")

Jumlah gambar terindikasi sangat buram: 265 dari 26462


In [82]:
print("\n10 gambar dengan skor blur terendah:")
print(df_blur_clean.nsmallest(10, 'blur_score')[['file_path', 'blur_score']].to_string(index=False))


10 gambar dengan skor blur terendah:
                          file_path  blur_score
      train/0_Recyclable/R_6393.jpg    0.487546
      train/0_Recyclable/R_9497.jpg    0.516385
      train/0_Recyclable/R_3441.jpg    0.652444
      train/0_Recyclable/R_6313.jpg    0.865304
      train/0_Recyclable/R_6483.jpg    0.961455
      train/0_Recyclable/R_3487.jpg    0.992567
      train/0_Recyclable/R_3420.jpg    1.091792
         train/2_Organic/O_5263.jpg    1.108332
train/1_Electronic/Keyboard_121.jpg    1.150711
      train/0_Recyclable/R_6386.jpg    1.183932


Semakin Kecil = Semaikin Buram

In [83]:
df_blur_clean

,id,label,label_number,label_folder_name,file_name,file_path,not_corrupt,blur_score,is_blurry
0,R_1,Recyclable,0,0_Recyclable,R_1.jpg,train/0_Recyclable/R_1.jpg,True,650.328699,False
1,R_2,Recyclable,0,0_Recyclable,R_2.jpg,train/0_Recyclable/R_2.jpg,True,302.200695,False
2,R_3,Recyclable,0,0_Recyclable,R_3.jpg,train/0_Recyclable/R_3.jpg,True,293.291705,False
3,R_4,Recyclable,0,0_Recyclable,R_4.jpg,train/0_Recyclable/R_4.jpg,True,2898.406776,False
4,R_5,Recyclable,0,0_Recyclable,R_5.jpg,train/0_Recyclable/R_5.jpg,True,206.334178,False
...,...,...,...,...,...,...,...,...,...
26519,O_12563,Organic,2,2_Organic,O_12563.jpg,train/2_Organic/O_12563.jpg,True,3883.561468,False
26520,O_12564,Organic,2,2_Organic,O_12564.jpg,train/2_Organic/O_12564.jpg,True,1372.350928,False
26521,O_12565,Organic,2,2_Organic,O_12565.jpg,train/2_Organic/O_12565.jpg,True,6362.614921,False
26522,O_12566,Organic,2,2_Organic,O_12566.jpg,train/2_Organic/O_12566.jpg,True,2693.580338,False


In [84]:
df_final = df_blur_clean[~df_blur_clean['is_blurry']].copy()
df_final.drop(columns=['blur_score', 'is_blurry'], inplace=True)

In [85]:
len(df_final)

26197

In [86]:
df_final.to_csv("dataset_cleaned.csv", index=False)